# Сравнение метрик жестикуляции

Это шаблонный ноутбук для сравнения нескольких способов оценки жестикуляции на заранее собранной выборке видео.

Ноутбук подготовлен как каркас. Сюда нужно будет добавить реальную логику извлечения признаков, прогнать её на настоящих роликах, а перед коммитом очистить outputs.

## Цели сравнения

Обязательно нужно сравнить:
1. Текущий baseline из Charisma Master.
2. Простой нормализованный метод, предложенный нейросетью.
3. Более сложный, но CPU-friendly метод, предложенный нейросетью.

Что должно получиться по итогам:
- хорошие профессиональные ролики должны уверенно отделяться от плохих по сырым метрикам;
- собственные ролики должны оказываться ближе к хорошим, чем к плохим;
- метрика не должна слишком сильно зависеть от расстояния до камеры;
- должно быть измерено время обработки как по каждому ролику, так и по всей выборке.

In [ ]:
from pathlib import Path
from time import perf_counter
from typing import Any

import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    import cv2
except ImportError:
    cv2 = None

try:
    import mediapipe as mp
except ImportError:
    mp = None


In [ ]:
PROJECT_ROOT = Path("/Users/mas562/PycharmProjects/metricconfig")
GESTURE_DIR = PROJECT_ROOT / "video_metrics" / "gesture"
MEDIA_DIR = GESTURE_DIR / "media"
GRAPHICS_DIR = MEDIA_DIR / "graphics"
RESULTS_DIR = GESTURE_DIR / "results"
MANIFEST_PATH = GESTURE_DIR / "dataset_manifest.csv"
COMPARISON_TABLE_PATH = GESTURE_DIR / "comparison_table.csv"
CONFIG_PATH = GESTURE_DIR / "config.json"

print('Папка gesture:', GESTURE_DIR)
print('Путь к manifest:', MANIFEST_PATH)
print('Папка с видео:', MEDIA_DIR)
print('Папка с графиками:', GRAPHICS_DIR)
print('Папка с результатами:', RESULTS_DIR)

## Загрузка manifest-файла

Ожидаются группы:
- `good_professional`
- `bad`
- `self_recorded`

In [ ]:
manifest_df = pd.read_csv(MANIFEST_PATH)
manifest_df

In [ ]:
required_columns = [
    'video_id',
    'имя_файла',
    'группа',
    'тип_источника',
    'длительность_сек',
    'исходный_fps',
    'исходное_разрешение',
    'категория_дистанции',
    'есть_показывание_на_слайды',
    'заметки',
]
missing_columns = [c for c in required_columns if c not in manifest_df.columns]
assert not missing_columns, f'Отсутствуют колонки в manifest: {missing_columns}'

manifest_df['video_path'] = manifest_df['имя_файла'].map(lambda name: str(MEDIA_DIR / name))
manifest_df[['video_id', 'группа', 'категория_дистанции', 'video_path']]

## Метод 1 — текущий baseline из Charisma Master

Референс в этом репозитории:
- `charisma-master/services/ml_worker/app/logic/ml_engine.py`
- `charisma-master/services/ml_worker/app/config.json`
- `charisma-master/services/ml_worker/app/config_fields.md`

Краткая логика текущего подхода:
- по кадрам накапливается движение при наличии распознанной позы,
- считается среднее движение,
- затем оно масштабируется через `gesture_scale`,
- результат ограничивается сверху `max_score`,
- дальше используются нижний и верхний пороги для интерпретации.

In [ ]:
CHARISMA_BASELINE_CONFIG = {
    'target_frame_width': 480,
    'frame_sample_every': 5,
    'min_frames_with_pose': 10,
    'movement_threshold': 0.002,
    'gesture_scale': 3500,
    'gesture_low_lt': 15,
    'gesture_high_gt': 85,
    'max_score': 100,
    'mediapipe': {
        'min_detection_confidence': 0.5,
        'min_tracking_confidence': 0.5,
        'model_complexity': 1,
        'static_image_mode': False,
    },
    'landmarks': {
        'left_shoulder': 11,
        'right_shoulder': 12,
        'left_elbow': 13,
        'right_elbow': 14,
        'left_wrist': 15,
        'right_wrist': 16,
    },
}
CHARISMA_BASELINE_CONFIG

In [ ]:
def ensure_video_dependencies() -> None:
    if cv2 is None:
        raise ImportError('OpenCV (cv2) не установлен. Установите opencv-python для анализа видео.')
    if mp is None:
        raise ImportError('MediaPipe не установлен. Установите mediapipe для анализа жестикуляции.')


def empty_metric_result(start: float, **extra: Any) -> dict[str, Any]:
    return {
        'raw_metric': np.nan,
        'score': np.nan,
        'processing_time_sec': perf_counter() - start,
        **extra,
    }


def euclidean_distance(point_a: tuple[float, float], point_b: tuple[float, float]) -> float:
    return float(((point_a[0] - point_b[0]) ** 2 + (point_a[1] - point_b[1]) ** 2) ** 0.5)


def clip_score(value: float, max_score: float = 100.0) -> float:
    return float(np.clip(value, 0.0, max_score))


def extract_pose_timeseries(video_path: str, config: dict[str, Any]) -> dict[str, Any]:
    ensure_video_dependencies()

    start = perf_counter()
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return {
            'frames_processed': 0,
            'frames_sampled': 0,
            'frames_with_pose': 0,
            'fps': np.nan,
            'frame_count': 0,
            'width': 0,
            'height': 0,
            'sampled_fps': np.nan,
            'records': [],
            'processing_time_sec': perf_counter() - start,
        }

    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH) or 0)
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT) or 0)

    mp_holistic = mp.solutions.holistic
    mp_pose = mp.solutions.pose
    mediapipe_config = config['mediapipe']
    frame_sample_every = max(int(config['frame_sample_every']), 1)
    target_frame_width = int(config['target_frame_width'])

    records: list[dict[str, Any]] = []
    frames_processed = 0
    frames_sampled = 0
    frames_with_pose = 0

    try:
        with mp_holistic.Holistic(
            min_detection_confidence=mediapipe_config['min_detection_confidence'],
            min_tracking_confidence=mediapipe_config['min_tracking_confidence'],
            model_complexity=mediapipe_config['model_complexity'],
            static_image_mode=mediapipe_config['static_image_mode'],
        ) as holistic:
            while True:
                ret, frame = cap.read()
                if not ret:
                    break

                frames_processed += 1
                if frames_processed % frame_sample_every != 0:
                    continue

                frames_sampled += 1
                if width > 0 and target_frame_width > 0:
                    scale_mp = target_frame_width / width
                    frame = cv2.resize(frame, (0, 0), fx=scale_mp, fy=scale_mp)

                img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                img_rgb.flags.writeable = False
                results = holistic.process(img_rgb)
                img_rgb.flags.writeable = True

                if not results.pose_landmarks:
                    continue

                pose_landmarks = results.pose_landmarks.landmark
                landmarks = config['landmarks']

                left_shoulder = pose_landmarks[landmarks['left_shoulder']]
                right_shoulder = pose_landmarks[landmarks['right_shoulder']]
                left_elbow = pose_landmarks[landmarks['left_elbow']]
                right_elbow = pose_landmarks[landmarks['right_elbow']]
                left_wrist = pose_landmarks[landmarks['left_wrist']]
                right_wrist = pose_landmarks[landmarks['right_wrist']]

                shoulder_width = euclidean_distance(
                    (left_shoulder.x, left_shoulder.y),
                    (right_shoulder.x, right_shoulder.y),
                )

                if shoulder_width <= 0:
                    continue

                frames_with_pose += 1
                records.append({
                    'frame_index': frames_processed,
                    'time_sec': frames_processed / fps if fps else np.nan,
                    'left_shoulder': (left_shoulder.x, left_shoulder.y),
                    'right_shoulder': (right_shoulder.x, right_shoulder.y),
                    'left_elbow': (left_elbow.x, left_elbow.y),
                    'right_elbow': (right_elbow.x, right_elbow.y),
                    'left_wrist': (left_wrist.x, left_wrist.y),
                    'right_wrist': (right_wrist.x, right_wrist.y),
                    'left_wrist_visibility': float(left_wrist.visibility),
                    'right_wrist_visibility': float(right_wrist.visibility),
                    'shoulder_width': shoulder_width,
                    'hip_center_x': float((left_shoulder.x + right_shoulder.x) / 2),
                    'hip_center_y': float((left_shoulder.y + right_shoulder.y) / 2),
                    'pose_landmarks': pose_landmarks,
                })
    finally:
        cap.release()

    sampled_fps = fps / frame_sample_every if frame_sample_every else fps
    return {
        'frames_processed': frames_processed,
        'frames_sampled': frames_sampled,
        'frames_with_pose': frames_with_pose,
        'fps': fps,
        'frame_count': frame_count,
        'width': width,
        'height': height,
        'sampled_fps': sampled_fps,
        'records': records,
        'processing_time_sec': perf_counter() - start,
    }


def build_normalized_frame_features(records: list[dict[str, Any]]) -> list[dict[str, Any]]:
    features: list[dict[str, Any]] = []
    previous = None

    for record in records:
        shoulder_width = record['shoulder_width']
        if shoulder_width <= 0:
            previous = None
            continue

        left_wrist = record['left_wrist']
        right_wrist = record['right_wrist']
        left_shoulder = record['left_shoulder']
        right_shoulder = record['right_shoulder']
        left_elbow = record['left_elbow']
        right_elbow = record['right_elbow']
        torso_center = (
            (left_shoulder[0] + right_shoulder[0]) / 2,
            (left_shoulder[1] + right_shoulder[1]) / 2,
        )

        hand_speed_norm = np.nan
        if previous is not None:
            delta_left = euclidean_distance(left_wrist, previous['left_wrist'])
            delta_right = euclidean_distance(right_wrist, previous['right_wrist'])
            hand_speed_norm = (delta_left + delta_right) / shoulder_width

        left_amplitude = euclidean_distance(left_wrist, torso_center) / shoulder_width
        right_amplitude = euclidean_distance(right_wrist, torso_center) / shoulder_width
        hand_amplitude_norm = float((left_amplitude + right_amplitude) / 2)

        left_horizontal_extension = abs(left_wrist[0] - left_shoulder[0]) / shoulder_width
        right_horizontal_extension = abs(right_wrist[0] - right_shoulder[0]) / shoulder_width
        left_pointing = left_horizontal_extension > 0.9 and left_wrist[1] <= left_elbow[1] + 0.05
        right_pointing = right_horizontal_extension > 0.9 and right_wrist[1] <= right_elbow[1] + 0.05
        pointing_flag = float(left_pointing or right_pointing)

        features.append({
            **record,
            'hand_speed_norm': float(hand_speed_norm) if not np.isnan(hand_speed_norm) else np.nan,
            'hand_amplitude_norm': hand_amplitude_norm,
            'pointing_flag': pointing_flag,
        })
        previous = record

    return features


def detect_gesture_episodes(features: list[dict[str, Any]], active_threshold: float, max_gap_frames: int = 1) -> list[list[dict[str, Any]]]:
    episodes: list[list[dict[str, Any]]] = []
    current_episode: list[dict[str, Any]] = []
    gap_count = 0

    for feature in features:
        speed = feature['hand_speed_norm']
        is_active = not np.isnan(speed) and speed > active_threshold

        if is_active:
            if gap_count and current_episode:
                gap_count = 0
            current_episode.append(feature)
            continue

        if current_episode:
            if gap_count < max_gap_frames:
                gap_count += 1
                current_episode.append(feature)
            else:
                episodes.append(current_episode)
                current_episode = []
                gap_count = 0

    if current_episode:
        episodes.append(current_episode)

    return [episode for episode in episodes if any((not np.isnan(item['hand_speed_norm']) and item['hand_speed_norm'] > active_threshold) for item in episode)]


In [ ]:
def run_charisma_master_baseline(video_path: str, config: dict[str, Any]) -> dict[str, Any]:
    start = perf_counter()
    pose_data = extract_pose_timeseries(video_path, config)
    records = pose_data['records']
    frames_with_pose = pose_data['frames_with_pose']

    if frames_with_pose <= config['min_frames_with_pose']:
        return empty_metric_result(
            start,
            frames_processed=pose_data['frames_processed'],
            frames_with_pose=frames_with_pose,
            movement_accum=0.0,
        )

    movement_accum = 0.0
    prev_left = None
    prev_right = None

    for record in records:
        left_wrist = record['left_wrist']
        right_wrist = record['right_wrist']

        if prev_left is not None and prev_right is not None:
            d_left = euclidean_distance(left_wrist, prev_left)
            d_right = euclidean_distance(right_wrist, prev_right)
            delta = d_left + d_right
            if delta > config['movement_threshold']:
                movement_accum += delta

        prev_left = left_wrist
        prev_right = right_wrist

    avg_move = movement_accum / frames_with_pose
    gesture_score = min(avg_move * config['gesture_scale'], config['max_score'])

    return {
        'raw_metric': float(avg_move),
        'score': float(gesture_score),
        'frames_processed': pose_data['frames_processed'],
        'frames_with_pose': frames_with_pose,
        'movement_accum': float(movement_accum),
        'processing_time_sec': perf_counter() - start,
    }


## Метод 2 — простой нормализованный метод

Возможные базовые признаки:
- нормализованная скорость движения рук относительно ширины плеч,
- амплитуда жестов относительно масштаба корпуса,
- доля кадров с активной жестикуляцией,
- при необходимости — доля кадров с жестами, похожими на показывание на слайды.

Цель: добиться лучшего разделения, чем у baseline, но сохранить простоту и скорость.

In [ ]:
def run_simple_normalized_method(video_path: str) -> dict[str, Any]:
    start = perf_counter()
    pose_data = extract_pose_timeseries(video_path, CHARISMA_BASELINE_CONFIG)
    features = build_normalized_frame_features(pose_data['records'])

    if pose_data['frames_with_pose'] <= CHARISMA_BASELINE_CONFIG['min_frames_with_pose'] or not features:
        return empty_metric_result(
            start,
            activity_ratio=np.nan,
            amplitude_mean=np.nan,
            pointing_ratio=np.nan,
        )

    speed_threshold = 0.08
    valid_speeds = [item['hand_speed_norm'] for item in features if not np.isnan(item['hand_speed_norm'])]
    active_flags = [speed > speed_threshold for speed in valid_speeds]
    activity_ratio = float(np.mean(active_flags)) if active_flags else 0.0
    amplitude_mean = float(np.mean([item['hand_amplitude_norm'] for item in features]))
    pointing_ratio = float(np.mean([item['pointing_flag'] for item in features]))

    raw_metric = (
        1.8 * activity_ratio
        + 0.9 * amplitude_mean
        + 0.35 * pointing_ratio
    )
    score = clip_score(raw_metric * 30.0)

    return {
        'raw_metric': float(raw_metric),
        'score': score,
        'activity_ratio': activity_ratio,
        'amplitude_mean': amplitude_mean,
        'pointing_ratio': pointing_ratio,
        'processing_time_sec': perf_counter() - start,
    }


## Метод 3 — более сложный, но CPU-friendly метод

Что можно добавить по сравнению с простым методом:
- сегментацию на эпизоды жестикуляции,
- анализ ритма и структуры пауз,
- анализ вариативности и повторяемости движений,
- штрафы за чрезмерную статичность или чрезмерную хаотичность.

Цель: получить более содержательный сигнал, оставаясь в пределах нормальной работы на CPU.

In [ ]:
def run_cpu_structured_method(video_path: str) -> dict[str, Any]:
    start = perf_counter()
    pose_data = extract_pose_timeseries(video_path, CHARISMA_BASELINE_CONFIG)
    features = build_normalized_frame_features(pose_data['records'])

    if pose_data['frames_with_pose'] <= CHARISMA_BASELINE_CONFIG['min_frames_with_pose'] or not features:
        return empty_metric_result(
            start,
            episode_count=np.nan,
            episode_duration_mean=np.nan,
            rhythm_score=np.nan,
        )

    speed_threshold = 0.08
    episodes = detect_gesture_episodes(features, active_threshold=speed_threshold, max_gap_frames=1)
    sampled_fps = pose_data['sampled_fps'] if not np.isnan(pose_data['sampled_fps']) else 5.0

    valid_speeds = np.array([item['hand_speed_norm'] for item in features if not np.isnan(item['hand_speed_norm'])], dtype=float)
    amplitudes = np.array([item['hand_amplitude_norm'] for item in features], dtype=float)
    pointing_values = np.array([item['pointing_flag'] for item in features], dtype=float)

    activity_ratio = float(np.mean(valid_speeds > speed_threshold)) if len(valid_speeds) else 0.0
    amplitude_mean = float(np.mean(amplitudes)) if len(amplitudes) else 0.0
    pointing_ratio = float(np.mean(pointing_values)) if len(pointing_values) else 0.0

    episode_count = len(episodes)
    episode_durations = [len(ep) / sampled_fps for ep in episodes if sampled_fps]
    episode_duration_mean = float(np.mean(episode_durations)) if episode_durations else 0.0

    episode_intensities = []
    episode_amplitudes = []
    episode_centers = []
    for ep in episodes:
        ep_speeds = [item['hand_speed_norm'] for item in ep if not np.isnan(item['hand_speed_norm'])]
        ep_amplitudes = [item['hand_amplitude_norm'] for item in ep]
        if ep_speeds:
            episode_intensities.append(float(np.mean(ep_speeds)))
        if ep_amplitudes:
            episode_amplitudes.append(float(np.mean(ep_amplitudes)))
        active_times = [item['time_sec'] for item in ep if not np.isnan(item['hand_speed_norm']) and item['hand_speed_norm'] > speed_threshold]
        if active_times:
            episode_centers.append(float(np.mean(active_times)))

    variability_score = 0.0
    if len(episode_amplitudes) >= 2:
        variability_score = float(np.std(episode_amplitudes))

    rhythm_score = 0.0
    if len(episode_centers) >= 3:
        intervals = np.diff(episode_centers)
        interval_mean = float(np.mean(intervals)) if len(intervals) else 0.0
        if interval_mean > 0:
            rhythm_score = float(1.0 / (1.0 + (np.std(intervals) / interval_mean)))

    stillness_penalty = max(0.0, 0.3 - activity_ratio)
    chaos_penalty = max(0.0, float(np.std(valid_speeds) - 0.18)) if len(valid_speeds) else 0.0

    raw_metric = (
        1.3 * activity_ratio
        + 0.8 * amplitude_mean
        + 0.7 * rhythm_score
        + 0.8 * variability_score
        + 0.25 * pointing_ratio
        - 0.8 * stillness_penalty
        - 0.4 * chaos_penalty
    )
    score = clip_score(raw_metric * 28.0)

    return {
        'raw_metric': float(raw_metric),
        'score': score,
        'episode_count': episode_count,
        'episode_duration_mean': float(episode_duration_mean),
        'rhythm_score': float(rhythm_score),
        'processing_time_sec': perf_counter() - start,
    }


## Пакетный запуск

Ниже — каркас для запуска всех методов на всех роликах с сохранением значений метрик и времени обработки.

In [ ]:
METHODS = {
    'charisma_master_baseline': lambda video_path: run_charisma_master_baseline(video_path, CHARISMA_BASELINE_CONFIG),
    'llm_simple_normalized': run_simple_normalized_method,
    'llm_cpu_structured': run_cpu_structured_method,
}

def run_all_methods(manifest: pd.DataFrame) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    for item in manifest.to_dict(orient='records'):
        for method_name, method_fn in METHODS.items():
            metrics = method_fn(item['video_path'])
            rows.append({
                'video_id': item['video_id'],
                'имя_файла': item['имя_файла'],
                'группа': item['группа'],
                'категория_дистанции': item['категория_дистанции'],
                'метод': method_name,
                **metrics,
            })
    return pd.DataFrame(rows)

# results_df = run_all_methods(manifest_df)
# results_df.head()

## Стандартизированный разрыв

Главная обязательная метрика сравнения:
- стандартизированный разрыв между `good_professional` и `bad` по сырой метрике.

В качестве базового варианта можно использовать аналог effect size с объединённым стандартным отклонением.

In [ ]:
def standardized_gap(good_values: pd.Series, bad_values: pd.Series) -> float:
    good = pd.to_numeric(good_values, errors='coerce').dropna()
    bad = pd.to_numeric(bad_values, errors='coerce').dropna()
    if len(good) < 2 or len(bad) < 2:
        return float('nan')

    good_mean = good.mean()
    bad_mean = bad.mean()
    good_var = good.var(ddof=1)
    bad_var = bad.var(ddof=1)
    pooled_std = math.sqrt(((len(good) - 1) * good_var + (len(bad) - 1) * bad_var) / (len(good) + len(bad) - 2))
    if pooled_std == 0:
        return float('nan')
    return float((good_mean - bad_mean) / pooled_std)


## Построение итоговой сравнительной таблицы

Ожидаемые колонки:
- метод
- описание метода
- стандартизированный разрыв good vs bad
- общее время обработки
- время обработки по роликам
- при желании — плюсы и минусы

In [ ]:
METHOD_DESCRIPTIONS = {
    'charisma_master_baseline': 'Текущий baseline Charisma Master на основе среднего движения рук, масштабированного в score.',
    'llm_simple_normalized': 'Простой нормализованный метод на основе активности жестикуляции, амплитуды и масштабно-нормализованного движения.',
    'llm_cpu_structured': 'Более сложный CPU-friendly метод на основе эпизодов жестикуляции, ритма и вариативности движений.',
}

TIME_COLUMN_MAP = {
    'prof_01': 'time_prof_01_sec',
    'prof_02': 'time_prof_02_sec',
    'prof_03': 'time_prof_03_sec',
    'prof_04': 'time_prof_04_sec',
    'bad_01': 'time_bad_01_sec',
    'bad_02': 'time_bad_02_sec',
    'self_01': 'time_self_01_sec',
    'self_02': 'time_self_02_sec',
    'self_03': 'time_self_03_sec',
}

def build_comparison_table(results_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for method_name, method_df in results_df.groupby('метод'):
        good_values = method_df.loc[method_df['группа'] == 'good_professional', 'raw_metric']
        bad_values = method_df.loc[method_df['группа'] == 'bad', 'raw_metric']
        gap = standardized_gap(good_values, bad_values)
        total_time = pd.to_numeric(method_df['processing_time_sec'], errors='coerce').sum()
        per_video_times = (
            method_df.groupby('video_id')['processing_time_sec']
            .first()
            .apply(lambda value: float(pd.to_numeric(value, errors='coerce')) if pd.notna(pd.to_numeric(value, errors='coerce')) else np.nan)
            .to_dict()
        )

        row = {
            'метод': method_name,
            'описание_метода': METHOD_DESCRIPTIONS.get(method_name, ''),
            'стандартизированный_разрыв_good_vs_bad': gap,
            'общее_время_обработки_сек': total_time,
            'плюсы': '',
            'минусы': '',
        }
        for video_id, column_name in TIME_COLUMN_MAP.items():
            row[column_name] = per_video_times.get(video_id, np.nan)
        rows.append(row)

    ordered_columns = [
        'метод',
        'описание_метода',
        'стандартизированный_разрыв_good_vs_bad',
        'общее_время_обработки_сек',
        *TIME_COLUMN_MAP.values(),
        'плюсы',
        'минусы',
    ]
    return pd.DataFrame(rows)[ordered_columns].sort_values('стандартизированный_разрыв_good_vs_bad', ascending=False)

# comparison_df = build_comparison_table(results_df)
# comparison_df

## Анализ чувствительности к расстоянию

Здесь нужно проверить, насколько стабильно ведёт себя метрика на `self_recorded` роликах, записанных на `near`, `mid` и `far`, если манера речи и жестикуляции остаётся примерно одинаковой.

In [ ]:
def analyze_distance_sensitivity(results_df: pd.DataFrame) -> pd.DataFrame:
    self_df = results_df[results_df['группа'] == 'self_recorded'].copy()
    summary_rows = []
    for method_name, method_df in self_df.groupby('метод'):
        pivot = method_df.pivot_table(index='video_id', columns='категория_дистанции', values='raw_metric', aggfunc='first')
        near_mid_far_spread = float('nan')
        if not pivot.empty:
            near_mid_far_spread = float((pivot.max(axis=1) - pivot.min(axis=1)).mean())
        summary_rows.append({
            'метод': method_name,
            'средний_разброс_на_собственных_роликах': near_mid_far_spread,
        })
    return pd.DataFrame(summary_rows).sort_values('средний_разброс_на_собственных_роликах')

# distance_df = analyze_distance_sensitivity(results_df)
# distance_df

## Выбор алгоритма и подбор порогов

Когда появятся реальные результаты, финальный метод нужно выбирать по совокупности факторов:
- насколько велик и полезен стандартизированный разрыв между good и bad;
- приемлемо ли время работы на CPU;
- хорошо ли ведут себя собственные ролики;
- не слишком ли чувствительна метрика к расстоянию до камеры.

После этого результаты нужно обсудить с тимлидом и заполнить `config.json` и `config.md`.

## Визуализация результатов

После расчёта статистики полезно сразу строить графики, чтобы быстрее сравнивать методы и экспортировать отчётные PNG в `media/graphics/`.

Минимальный набор графиков:
- сырые метрики по роликам и методам;
- standardized gap по методам;
- общее время обработки по методам;
- чувствительность к расстоянию по методам.

In [ ]:
def plot_raw_metrics_by_method(results_df: pd.DataFrame):
    pivot = results_df.pivot_table(index='video_id', columns='метод', values='raw_metric', aggfunc='first')
    fig, ax = plt.subplots(figsize=(12, 6))
    pivot.plot(kind='bar', ax=ax)
    ax.set_title('Сырые метрики жестикуляции по роликам и методам')
    ax.set_xlabel('video_id')
    ax.set_ylabel('raw_metric')
    ax.legend(title='Метод')
    ax.grid(axis='y', alpha=0.3)
    fig.tight_layout()
    return fig


def plot_standardized_gap(comparison_df: pd.DataFrame):
    plot_df = comparison_df[['метод', 'стандартизированный_разрыв_good_vs_bad']].copy()
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(plot_df['метод'], plot_df['стандартизированный_разрыв_good_vs_bad'])
    ax.set_title('Стандартизированный разрыв good vs bad')
    ax.set_xlabel('Метод')
    ax.set_ylabel('standardized_gap')
    ax.tick_params(axis='x', rotation=20)
    ax.grid(axis='y', alpha=0.3)
    fig.tight_layout()
    return fig


def plot_total_processing_time(comparison_df: pd.DataFrame):
    plot_df = comparison_df[['метод', 'общее_время_обработки_сек']].copy()
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(plot_df['метод'], plot_df['общее_время_обработки_сек'])
    ax.set_title('Общее время обработки по методам')
    ax.set_xlabel('Метод')
    ax.set_ylabel('seconds')
    ax.tick_params(axis='x', rotation=20)
    ax.grid(axis='y', alpha=0.3)
    fig.tight_layout()
    return fig


def plot_distance_sensitivity(distance_df: pd.DataFrame):
    plot_df = distance_df[['метод', 'средний_разброс_на_собственных_роликах']].copy()
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(plot_df['метод'], plot_df['средний_разброс_на_собственных_роликах'])
    ax.set_title('Чувствительность к расстоянию')
    ax.set_xlabel('Метод')
    ax.set_ylabel('spread')
    ax.tick_params(axis='x', rotation=20)
    ax.grid(axis='y', alpha=0.3)
    fig.tight_layout()
    return fig


def export_graphics(results_df: pd.DataFrame, comparison_df: pd.DataFrame, distance_df: pd.DataFrame) -> None:
    GRAPHICS_DIR.mkdir(parents=True, exist_ok=True)

    figures = {
        'raw_metrics_by_method.png': plot_raw_metrics_by_method(results_df),
        'standardized_gap_by_method.png': plot_standardized_gap(comparison_df),
        'processing_time_by_method.png': plot_total_processing_time(comparison_df),
        'distance_sensitivity_by_method.png': plot_distance_sensitivity(distance_df),
    }

    for file_name, fig in figures.items():
        fig.savefig(GRAPHICS_DIR / file_name, dpi=150, bbox_inches='tight')
        plt.close(fig)


def export_results(results_df: pd.DataFrame, comparison_df: pd.DataFrame, distance_df: pd.DataFrame | None = None) -> None:
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    results_df.to_csv(RESULTS_DIR / 'per_video_metrics.csv', index=False)
    comparison_df.to_csv(COMPARISON_TABLE_PATH, index=False)

    if distance_df is not None:
        distance_df.to_csv(RESULTS_DIR / 'distance_sensitivity.csv', index=False)
        export_graphics(results_df, comparison_df, distance_df)

    print(f'Сохранены результаты в: {RESULTS_DIR}')
    print(f'Обновлена таблица сравнения: {COMPARISON_TABLE_PATH}')


results_df = run_all_methods(manifest_df)
comparison_df = build_comparison_table(results_df)
distance_df = analyze_distance_sensitivity(results_df)
export_results(results_df, comparison_df, distance_df)

results_df.head(), comparison_df, distance_df

## Финальный чек-лист

Перед завершением работы в целевом проекте нужно проверить:
- [ ] Все реальные ролики добавлены в `media/`.
- [ ] `dataset_manifest.csv` заполнен.
- [ ] Все три метода реализованы.
- [ ] Сравнение запущено на всей выборке.
- [ ] `comparison_table.csv` заполнен.
- [ ] Лучший алгоритм и пороги обсуждены с тимлидом.
- [ ] `config.json` заполнен.
- [ ] Обоснование занесено в `config.md`.
- [ ] Outputs ноутбука очищены перед коммитом.